# 02 — Sentiment Analysis

Runs the sentiment question over entries (resumable), then plots:
- sentiment distribution by day of week
- sentiment distribution by time of day
- weekly average sentiment score over time

**Smoke mode:** This notebook runs sentiment over the most recent `LIMIT` entries. Set `LIMIT = None` to process the entire corpus (takes ~25 min).

In [1]:
import sys, json
sys.path.insert(0, "../src")

import pandas as pd
import plotly.express as px
from IPython.display import display

from journal.config import LANCE_ROOT, LLM_MODEL
from journal.analyze import BatchAnalyzer, QuestionRegistry
from journal.llm import OllamaLLM
from journal.questions.sentiment import SENTIMENT
from journal.store import Store

LIMIT = 200  # set to None for full corpus run (~25 min)

store = Store(LANCE_ROOT)

# Determine which entries to analyze
entries_df = store.entries_to_pandas()
if LIMIT is not None:
    entries_df = entries_df.sort_values("date").tail(LIMIT)
target_ids = set(entries_df["id"].tolist())
print(f"target entries: {len(target_ids)}")

# We need BatchAnalyzer to only process these. The simplest approach: temporarily
# filter the store via a custom analysis runner.
# Easier: monkey-patch entries_missing_analysis for this run.

original_missing = store.entries_missing_analysis
def scoped_missing(question_id, model):
    all_missing = original_missing(question_id, model)
    return [eid for eid in all_missing if eid in target_ids]
store.entries_missing_analysis = scoped_missing

reg = QuestionRegistry()
reg.register(SENTIMENT)
analyzer = BatchAnalyzer(store=store, llm=OllamaLLM(), registry=reg, model=LLM_MODEL, max_workers=4)
report = analyzer.run("sentiment")
print(report)

target entries: 200


AnalyzeReport(question_id='sentiment', processed=199, failed=1, errors=[])


In [2]:
adf = store.analyses_to_pandas()
adf = adf[adf["question_id"] == "sentiment"].copy()
adf = adf[adf["parsed_ok"]].copy()
adf["level"] = adf["result_json"].apply(lambda s: json.loads(s)["level"])
adf["score"] = adf["result_json"].apply(lambda s: json.loads(s)["score"])

edf = store.entries_to_pandas()[["id", "date", "day_of_week", "time_of_day"]]
df = adf.merge(edf, left_on="entry_id", right_on="id")
print(f"joined rows: {len(df)}")
df.head()

joined rows: 199


,entry_id,question_id,question_text,result_json,parsed_ok,model,created_at,level,score,id,date,day_of_week,time_of_day
0,b15e08a513844d19,sentiment,Respond with ONLY a JSON object of the form:\n...,"{""level"": ""negative"", ""score"": 3, ""confidence""...",True,gemma3:4b,2026-06-30 22:46:49.762185,negative,3,b15e08a513844d19,2026-04-09,Thu,night
1,c4f6d3864e7b24de,sentiment,Respond with ONLY a JSON object of the form:\n...,"{""level"": ""negative"", ""score"": 3, ""confidence""...",True,gemma3:4b,2026-06-30 22:46:50.636991,negative,3,c4f6d3864e7b24de,2026-04-07,Tue,night
2,62a2339631e6adf1,sentiment,Respond with ONLY a JSON object of the form:\n...,"{""level"": ""neutral"", ""score"": 3, ""confidence"":...",True,gemma3:4b,2026-06-30 22:46:52.566711,neutral,3,62a2339631e6adf1,2026-04-08,Wed,night
3,fbf397b1f4b7aa51,sentiment,Respond with ONLY a JSON object of the form:\n...,"{""level"": ""negative"", ""score"": 3, ""confidence""...",True,gemma3:4b,2026-06-30 22:46:54.788154,negative,3,fbf397b1f4b7aa51,2026-04-05,Sun,night
4,5b3ec6b02347706e,sentiment,Respond with ONLY a JSON object of the form:\n...,"{""level"": ""neutral"", ""score"": 2, ""confidence"":...",True,gemma3:4b,2026-06-30 22:46:56.309150,neutral,2,5b3ec6b02347706e,2026-04-08,Wed,night


In [3]:
order = ["extreme_negative", "negative", "neutral", "positive", "extreme_positive"]
day_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
ct = df.groupby(["day_of_week", "level"]).size().reset_index(name="count")
ct["day_of_week"] = pd.Categorical(ct["day_of_week"], categories=day_order, ordered=True)
fig = px.bar(ct, x="day_of_week", y="count", color="level",
             category_orders={"level": order},
             title="Sentiment distribution by day of week")
fig.show()

In [4]:
tod_order = ["morning", "afternoon", "evening", "night"]
ct = df.groupby(["time_of_day", "level"]).size().reset_index(name="count")
ct["time_of_day"] = pd.Categorical(ct["time_of_day"], categories=tod_order, ordered=True)
fig = px.bar(ct, x="time_of_day", y="count", color="level",
             category_orders={"level": order},
             title="Sentiment distribution by time of day")
fig.show()

In [5]:
df["date"] = pd.to_datetime(df["date"])
weekly = df.set_index("date")["score"].resample("W").mean().reset_index()
fig = px.line(weekly, x="date", y="score", title="Average sentiment score over time (weekly)")
fig.show()